# Model to analyze replacement of Russian imports by LNG

- no network conversion used
- analysis of the impact of LNG import capacity increasement

### Import packages

In [1]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import networkx as nx
import os
from infrastructure_model_data_import_functions import *
from infrastructure_model_input_data_prep_functions import *
from infrastructure_model_output_data_prep_functions import *

### Import Data

In [82]:
# Specify the path to your Excel file
input_file_path = os.path.join('..', '..','01_data', '01_input_data', '02_processed')
excel_file_name = 'data_case_russia_BASE.xlsx'
#data_case_russia_peak_hour.xlsx

In [83]:
# Call the function to load the data
df_nodes, df_commodities, df_edges, df_parameter, df_supply_values = load_input_data(input_file_path, excel_file_name)

In [84]:
(network_nodes, commodities, edges, initial_capacities, max_capacities, 
 edge_cost, pipe_new_cost, pipe_conv_cost, pipe_conv_factor, supply_values, 
 node_values) = extract_network_data(
     df_nodes, df_commodities, df_edges, df_parameter, df_supply_values)

### Create input data structure

In [85]:
'''
Create slack nodes for all supply nodes
link a node to supply in case of shortage in the system to all supply nodes
the capacity is infinite but at an infinite (super high) cost
'''
(shortage_list, shortage_edges_list, 
 shortage_capacity_dict, shortage_cost_dict) = create_slack_nodes_and_links(
     node_values, commodities)

In [86]:
'''
Create slack nodes for all supply nodes
enable excess nodes that oversupply is also not a problem
'''
(excess_list, excess_edges_list, 
excess_capacity_dict, excess_cost_dict) = create_excess_nodes_for_supply(
    node_values, commodities)

Check the graph

In [87]:
#check that all nodes are connected via edges
check_graph_connectivity(edges)

The graph is connected.


In [88]:
#check that all nodes are connected via edges and identify not connected components
check_graph_connectivity_and_components(edges)

The graph is connected.


In [89]:
# Check if all nodes are connected
check_if_all_nodes_connected(network_nodes, edges)

All nodes are connected.


## Model

### Create model

In [90]:
# Create a new model
model = gp.Model("Grid_Transformation")

### Define parameters

In [91]:
# Parameters
commodities = commodities  # Commodity types
#real network elements
network_nodes = network_nodes # Nodes of the system
network_edges = edges  # Edges
initial_capacities = initial_capacities # Initial capacities
max_capacities = max_capacities  # Maximum capacities
costs_edge = edge_cost  # Cost to transport from node to node
capacity_new_cost = pipe_new_cost  # Cost to increase capacity
capacity_change_cost = pipe_conv_cost  # Cost to increase capacity
node_value = node_values #contains supply and demand values

#implement factor to adjust capacity when conversion from methane to hydrogen
#TODO Implement it from the input file and use a correct factor
conversion_factor = pipe_conv_factor

#slack parameters shortage
shortage_nodes = shortage_list
shortage_edges = shortage_edges_list
shortage_capacities = shortage_capacity_dict
shortage_cost = shortage_cost_dict

#slack parameters excess
excess_nodes = excess_list
excess_edges = excess_edges_list
excess_capacities = excess_capacity_dict
excess_cost = excess_cost_dict

#complete network of the model
all_edges = network_edges + excess_edges_list + shortage_edges_list

### Define decision variables

In [92]:
# Decision variables
x_flow = {} #flow of commodity on an edge
x_flow_shortage = {}
x_flow_excess = {}
y_new_cap = {} #new build capacity for a commodity on an edge between two edges
z_conv_cap = {} #capacity of a commodity converted on an edge between two nodes
Change = {} # Binary variable for switching

for commodity in commodities:
    x_flow[commodity] = {}
    x_flow_shortage[commodity] = {}
    x_flow_excess[commodity] = {}
    y_new_cap[commodity] = {}
    z_conv_cap[commodity] = {}
    Change[commodity] = {}
    for edge in all_edges:
        x_flow[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"x_flow_{commodity}_{edge[0]}_{edge[1]}")
        y_new_cap[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"y_new_cap{commodity}_{edge[0]}_{edge[1]}")
        z_conv_cap[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"z_conv_cap{commodity}_{edge[0]}_{edge[1]}")
        Change[commodity][edge] = model.addVar(vtype=GRB.BINARY, name=f"change_{commodity}_{edge[0]}_{edge[1]}")
    for edge in shortage_edges:
        x_flow_shortage[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"x_flow_shortage_{commodity}_{edge[0]}_{edge[1]}")
    for edge in excess_edges:
        x_flow_excess[commodity][edge] = model.addVar(vtype=GRB.CONTINUOUS, name=f"x_flow_excess_{commodity}_{edge[0]}_{edge[1]}")
model.update()

In [93]:
network_edges_inc_excess = network_edges + excess_edges_list

### Define objective and constraints

In [94]:
# Objective function (minimize total transportation cost + cost to increase and convert capacity)
model.setObjective(
    gp.quicksum(x_flow[commodity][edge] * costs_edge[commodity][f"{edge[0]}{edge[1]}"] 
                for commodity in commodities for edge in network_edges) +
    #gp.quicksum(y_new_cap[commodity][edge] * capacity_new_cost[commodity][f"{edge[0]}{edge[1]}"] for commodity in commodities for edge in network_edges) +
    #gp.quicksum(z_conv_cap[commodity][edge] * capacity_change_cost[commodity][f"{edge[0]}{edge[1]}"] for commodity in commodities for edge in network_edges),
    gp.quicksum(x_flow_shortage[commodity][edge] * shortage_cost[commodity][f"{edge[0]}{edge[1]}"] 
                for commodity in commodities for edge in shortage_edges)+
    gp.quicksum(x_flow_excess[commodity][edge] * excess_cost[commodity][f"{edge[0]}{edge[1]}"] 
                for commodity in commodities for edge in excess_edges),
    GRB.MINIMIZE
)

# Constraints

#inflow of a node must equal the outflow of a node
for node in network_nodes:  
    for commodity in commodities:  
        # Flow conservation constraint for the current node and commodity
        model.addConstr(gp.quicksum(x_flow[commodity][edge] for edge in all_edges if edge[1] == node) 
                                    - gp.quicksum(x_flow[commodity][edge] for edge in network_edges_inc_excess if edge[0] == node)
                                    + node_value[commodity][node] 
                                    == 0, f"flow_constraint_{commodity}_{node}")

for node in excess_nodes:  
    for commodity in commodities:  
        # Flow conservation constraint for the current node and commodity
        model.addConstr(gp.quicksum(x_flow[commodity][edge] for edge in all_edges if edge[1] == node) 
                                    #- gp.quicksum(x_flow[commodity][edge] for edge in all_edges if edge[1] == node) 
                                    >= 0, f"flow_constraint_{commodity}_{node}")

# Add constraint for equality between x_flow and x_flow_shortage for the same edge
for commodity in commodities:
    for edge in shortage_edges:
        # Ensure equality for the corresponding edges
        model.addConstr(x_flow[commodity][edge] == x_flow_shortage[commodity][edge], 
                        f"equality_flow_shortage_constraint_{commodity}_{edge}")

#excess node at the sources to avoid infeasible problems.        
for node in excess_nodes:
    for commodity in commodities: 
        model.addConstr(gp.quicksum(x_flow[commodity][edge] for edge in excess_edges if edge[1] == node) >= 0,
                        f"supply_excess_{commodity}_{node}")


# Add constraint for equality between x_flow and x_flow_excess for the same edge
for commodity in commodities:
    for edge in excess_edges:
        # Ensure equality for the corresponding edges
        model.addConstr(x_flow[commodity][edge] == x_flow_excess[commodity][edge], 
                        f"equality_flow_excess_constraint_{commodity}_{edge}")

#Capacity constraint for excess flow
for commodity in commodities:
    for edge in excess_edges:
        model.addConstr(x_flow_excess[commodity][edge] 
                        <= 100000, 
                        f"capacity_limit_excess{commodity}_{edge[0]}_{edge[1]}")

#Capacity constraint for flow
for commodity in commodities:
    for edge in network_edges:
        model.addConstr(x_flow[commodity][edge] 
                        <= y_new_cap[commodity][edge] + z_conv_cap[commodity][edge] #+ initial_capacities[commodity][f"{edge[0]}{edge[1]}"]
                        , f"used_capacity_{commodity}_{edge[0]}_{edge[1]}")


# Capacity constraint for maximal capacity
for commodity in commodities:
    for edge in network_edges:
        model.addConstr(y_new_cap[commodity][edge] + z_conv_cap[commodity][edge] 
                        <= max_capacities[commodity][f"{edge[0]}{edge[1]}"], f"capacity_{commodity}_{edge[0]}_{edge[1]}")

#Constraints for edge conversion
for edge in network_edges:
    model.addConstr(Change[commodities[0]][edge] + Change[commodities[1]][edge] == 1, f"switching_constraint_{edge[0]}_{edge[1]}")

for commodity in commodities:
    for edge in network_edges:
        model.addConstr(initial_capacities[commodities[0]][f"{edge[0]}{edge[1]}"] * Change[commodity][edge] #* conversion_factor[commodity][node]
                        == z_conv_cap[commodity][edge], f"changed_capacity_{commodity}_{edge[0]}_{edge[1]}")

#Constraing no negative flow
for commodity in commodities:
    for edge in all_edges:
        model.addConstr(x_flow[commodity][edge] >= 0, f"non_negativity_x_{commodity}_{edge[0]}_{edge[1]}")

### Optimize the model

In [95]:
# Optimize the model
model.optimize()

Gurobi Optimizer version 11.0.0 build v11.0.0rc2 (win64 - Windows 11+.0 (26100.2))

CPU model: Intel(R) Core(TM) i5-8265U CPU @ 1.60GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Academic license 2606402 - for non-commercial use only - registered to jf___@cbs.dk
Optimize a model with 4174 rows, 3800 columns and 9000 nonzeros
Model fingerprint: 0x7cc24140
Variable types: 2892 continuous, 908 integer (908 binary)
Coefficient statistics:
  Matrix range     [8e-01, 1e+05]
  Objective range  [1e+00, 1e+07]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e-02, 1e+05]
Presolve removed 4072 rows and 3394 columns
Presolve time: 0.01s
Presolved: 102 rows, 406 columns, 731 nonzeros
Variable types: 406 continuous, 0 integer (0 binary)

Root relaxation: objective 4.638799e+07, 171 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth Int

### Results processing

In [96]:
#check model status
check_optimization_status(model)

Optimal solution found!


In [97]:
#call the function to store the results into DataFrames
results_df, excess_df, shortage_df = store_optimization_results(
    model,
    commodities,
    network_edges,
    excess_edges,
    shortage_edges,
    x_flow,
    y_new_cap,
    Change,
    z_conv_cap,
    x_flow_excess,
    x_flow_shortage
)

Get results

In [98]:
# Call the function to filter the DataFrames for each commodity
commodity_dfs = filter_commodities_to_dataframe(results_df, commodities)
methane_rows_df = commodity_dfs.get('Methane')

In [99]:
#create a duplicate for further operations
methane_results_df = methane_rows_df

In [100]:
#get initial capacities but in another format than above
initial_capacities_separator_data = create_initial_capacities_separator_dict(df_parameter)
initial_capacities_data = initial_capacities_separator_data

In [101]:
#get the share of use for each infrastructure element
methane_results_df = add_share_column(methane_results_df, initial_capacities_data)
methane_results_df

,Commodity,Edge,Flow,Share
0,Methane,"(BE_LNG, BE)",111.378000,1.000000
1,Methane,"(FR_LNG, FR)",322.410000,1.000000
2,Methane,"(EL_LNG, EL)",73.275000,1.000000
3,Methane,"(IT_LNG, IT)",155.831500,1.000000
4,Methane,"(HR_LNG, HR)",25.402000,1.000000
...,...,...,...,...
365,Methane,"(EG_LNG, EG)",0.000000,0.000000
366,Methane,"(SA_LNG, SA)",60.872293,0.006087
367,Methane,"(EG, EG_LNG)",0.000000,0.000000
368,Methane,"(CM, CM_LNG)",0.000000,0.000000


In [102]:
#aggregated share of capacity use
# Exclude flows and capacities from the following countries
excluded_countries = ['RU']
result_capacity_wo_ru_df = calculate_aggregated_capacity(methane_results_df, initial_capacities_data, excluded_countries)
result_capacity_wo_ru_df

,Commodity,Type,Total Flow,Total Capacity,Share
0,Methane,LNG,10760.795393,1.722577e+06,0.006247
1,Methane,Prod,33073.451279,5.200000e+05,0.063603
2,Methane,St,0.000000,0.000000e+00,0.000000


In [103]:
#aggregated share of supply use (e.g., from potential production, LNG or storage as a source)
# Exclude capacities and flows from Russia ('RU')
excluded_countries = ['RU']
result_supply_wo_ru_df = calculate_aggregated_share_supply(methane_results_df, node_values, excluded_countries)
result_supply_wo_ru_df

,Commodity,Type,Total Flow,Total Potential Supply,Share
0,Methane,LNG,10760.795393,0.000000,0.000000
1,Methane,Prod,33073.451279,36054.169034,0.917327
2,Methane,St,0.000000,0.000000,0.000000


In [104]:
search_element = 'RU'  # Example search element to look for in the 'Edge' column
commodity = 'Methane'  # Example commodity to filter by (optional)

# Call the function to filter the rows
filtered_df = filter_rows_by_search_element_and_commodity(results_df, 'Edge', search_element, commodity)
filtered_df

,Commodity,Edge,Flow,New Capacity,Switched,Changed Capacity
21,Methane,"(RU_Prod, RU)",4596.411157,10000.0000,0.0,0.0
86,Methane,"(LT, RU)",0.000000,41.6830,0.0,0.0
88,Methane,"(LV, RU)",0.000000,31.0250,0.0,0.0
97,Methane,"(RU, DE)",0.000000,635.8300,0.0,0.0
98,Methane,"(RU, LV)",0.000000,61.3200,0.0,0.0
99,Methane,"(RU, FI)",11.078905,80.3000,0.0,0.0
108,Methane,"(RU, UA)",240.336894,486.9100,0.0,0.0
110,Methane,"(RU, LV)",0.000000,61.3200,0.0,0.0
122,Methane,"(RU, BY)",0.000000,439.2775,0.0,0.0
147,Methane,"(RU, TR)",0.000000,10000.0000,0.0,0.0


In [80]:
search_element = 'RU'  # Example search element to look for in the 'Edge' column
commodity = 'Methane'  # Example commodity to filter by (optional)

# Call the function to filter the rows
filtered_df = filter_rows_by_search_element(methane_results_df, 'Edge', search_element)
filtered_df

,Commodity,Edge,Flow,Share
21,Methane,"(RU_Prod, RU)",2298.000521,0.229800
86,Methane,"(LT, RU)",8.871211,0.212826
88,Methane,"(LV, RU)",0.000000,0.000000
97,Methane,"(RU, DE)",0.000000,0.000000
98,Methane,"(RU, LV)",0.000000,0.000000
99,Methane,"(RU, FI)",19.540000,0.243337
108,Methane,"(RU, UA)",276.037392,0.566917
110,Methane,"(RU, LV)",0.000000,0.000000
122,Methane,"(RU, BY)",0.000000,0.000000
147,Methane,"(RU, TR)",0.000000,0.000000


In [26]:
#get just excess for methane out of the excess_df
methane_excess_df = excess_df[excess_df['Commodity'].str.contains('Methane', case=False)]

In [27]:
'''still not working'''
#check for an unused share of capacities
#aggregated share of supply use (e.g., from potential production, LNG or storage as a source)
# Exclude capacities and flows
excluded_countries = ['ES']
methane_excess_share_df = calculate_aggregated_share_supply(methane_excess_df, node_values, excluded_countries)
methane_excess_share_df

,Commodity,Type,Total Flow,Total Potential Supply,Share
0,Methane,LNG,0.000000,0.000000,0.000000
1,Methane,Prod,4621.930427,42291.343443,0.109288
2,Methane,St,0.000000,0.000000,0.000000


In [28]:
shortage_df

methane_shortage_df = shortage_df[shortage_df['Commodity'].str.contains('Methane', case=False)]
methane_shortage_df

,Commodity,Edge,Flow


In [ ]:
#methane_results_df.to_excel("output1.xlsx")